In [1]:

import json
import pandas as pd
from datasets import load_dataset, DatasetDict
from pprint import pprint
    
# The FEVER dataset is available on the Hugging Face Hub.
raw_dataset = load_dataset('fever', 'v2.0', trust_remote_code=True)
print(raw_dataset)

wiki = load_dataset('fever', 'wiki_pages', trust_remote_code=True)
print(wiki)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'fever' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since fever couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'v2.0' at C:\Users\minko\.cache\huggingface\datasets\fever\v2.0\2.0.0\7f8936e0558704771b08c7ce9cc202071b29a0050603374507ba61d23c00a58e (last modified on Tue Sep  9 23:21:51 2025).
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'fever' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


DatasetDict({
    validation: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 2384
    })
})


Using the latest cached version of the dataset since fever couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wiki_pages' at C:\Users\minko\.cache\huggingface\datasets\fever\wiki_pages\1.0.0\7f8936e0558704771b08c7ce9cc202071b29a0050603374507ba61d23c00a58e (last modified on Wed Sep 10 11:48:38 2025).


DatasetDict({
    wikipedia_pages: Dataset({
        features: ['id', 'text', 'lines'],
        num_rows: 5416537
    })
})


In [2]:
print(raw_dataset["train"][0])

KeyError: 'train'

In [ ]:
# print(wiki['wikipedia_pages'].filter(lambda example: example["id"] == "Nikolaj_Coster-Waldau"))

def get_sentence(wiki_id, sent_id):
    lines = wiki['wikipedia_pages'].filter(lambda example: example["id"] == wiki_id)[0]["lines"]
    for line in lines.split("\n"):
        sid, text = line.split("\t", 1)
        if int(sid) == sent_id:
            return text
    return None

print(get_sentence("Nikolaj_Coster-Waldau", 7))

In [2]:
from datasets import load_dataset, DatasetDict

# 1️⃣ Load the original dataset
dataset = load_dataset("minko186/fever-processed", split="train")

# Optional: shuffle before splitting
dataset = dataset.shuffle(seed=42)

# 2️⃣ Split into train and eval (80/20)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Combine into a DatasetDict for pushing
final_dataset = DatasetDict({
    "train": train_dataset,
    "eval": eval_dataset
})

print("Train size:", len(final_dataset["train"]))
print("Eval size:", len(final_dataset["eval"]))

# 3️⃣ Push to Hugging Face Hub
# You must be logged in: `huggingface-cli login`
final_dataset.push_to_hub("minko186/fever-processed", private=False)


Train size: 133887
Eval size: 33472


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/34 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/minko186/fever-processed/commit/9432d6a387d3a4957e0782360549bf6650f5d38d', commit_message='Upload dataset', commit_description='', oid='9432d6a387d3a4957e0782360549bf6650f5d38d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/minko186/fever-processed', endpoint='https://huggingface.co', repo_type='dataset', repo_id='minko186/fever-processed'), pr_revision=None, pr_num=None)

In [5]:
import torch
import yaml
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pprint import pprint


def load_config(config_path="config.yaml"):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config


def load_model(config, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model_id = config["training_args"].get("hub_model_id", config["model_name"])

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

    return tokenizer, model, device


def generate_claim(evidence_text, tokenizer, model, device, max_input_length=512, max_output_length=128):
    input_text = f"extract fact: {evidence_text} Based on this, what is the claim?"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_output_length,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

In [6]:
config = load_config("config.yaml")
tokenizer, model, device = load_model(config)
example_evidences = [
    # """Through the Looking-Glass, and What Alice Found There is a novel published in December 1871 by Lewis Carroll, the pen name of Charles Lutwidge Dodgson, a mathematics lecturer at Christ Church, Oxford. It is the sequel to his Alice's Adventures in Wonderland (1865), in which many of the characters were anthropomorphic playing cards.""",
    # """In this second novel, the theme is chess. As in the earlier book, the central figure, Alice, enters a fantastical world, this time by climbing through a large looking-glass (a mirror)[n 1] into a world that she can see beyond. There she finds that, just as in a reflection, things are reversed, including logic (for example, running helps one remain stationary, walking away from something brings one towards it, chessmen are alive and nursery-rhyme characters are real).""",
    # """Top Chef was launched on Bravo in 2006 and featured civilians called 'cheftestants' competing for $100,000, a feature in Food & Wine magazine, and a showcase at the Food & Wine Classic in Aspen.[1] The programme frequently had guests as judges, prompting the programme's judges Tom Colicchio and Hubert Keller to consider mounting a derivative of the programme for professionals.""",
    # """From 1986 to 1989, Karki worked as assistant teacher at Mahendra Multiple Campus, Dharan; from 1988, she concurrently was the bar president of the Koshi Zonal Court until 1990.[5][4] That year, she participated in the 1990 People's Movement to overthrow the Panchayat regime and was imprisoned in Biratnagar Jail.""",
    'Simone Biles made a triumphant return to the Olympic stage at the Paris 2024 Games, competing in the women’s gymnastics qualifications. Overcoming a previous struggle with the “twisties” that led to her withdrawal from events at the Tokyo 2020 Olympics, Biles dazzled with strong performances on all apparatus, helping the U.S. team secure a commanding lead in the qualifications. Her routines showcased her resilience and skill, drawing enthusiastic support from a star-studded audience',
]

for evidence in example_evidences:
    pprint(evidence)
    claim = generate_claim(evidence, tokenizer, model, device)
    print("Generated Claim:", claim)
    print("-" * 50)

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\minko\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minko\.cache\huggingface\hub\models--minko186--flan-t5-fact-extraction-fever-supports. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not install

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

('Simone Biles made a triumphant return to the Olympic stage at the Paris 2024 '
 'Games, competing in the women’s gymnastics qualifications. Overcoming a '
 'previous struggle with the “twisties” that led to her withdrawal from events '
 'at the Tokyo 2020 Olympics, Biles dazzled with strong performances on all '
 'apparatus, helping the U.S. team secure a commanding lead in the '
 'qualifications. Her routines showcased her resilience and skill, drawing '
 'enthusiastic support from a star-studded audience')
Generated Claim: Simone Biles is a gymnast. || Simone Biles is a person.
--------------------------------------------------


In [1]:
from datasets import load_dataset

try:
    dataset = load_dataset("polygraf-ai/enron-mails", split="train", download_mode="force_redownload")
except Exception as e:
    print("Failed to load dataset:", e)


README.md:   0%|          | 0.00/956 [00:00<?, ?B/s]

C:\Users\minko\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minko\.cache\huggingface\hub\datasets--polygraf-ai--enron-mails. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to re

train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/115374 [00:00<?, ? examples/s]

In [5]:
import re
import random
import datasets
import json
from tqdm import tqdm
from nltk import sent_tokenize
from cleaning import remove_special_characters
from content_hash import generate_content_hash_id, upload_dataset_to_collection


hub_name = "polygraf-ai/aid-enron-mails"
min_len = 0

empty_dataset = datasets.Dataset.from_dict(
    {"doc_id": [], "text": [], "metadata": [], "model_name": [], "domain": [], "link": []}
)

# Push the empty dataset to the Hugging Face Hub
empty_dataset.push_to_hub(hub_name, private=True)


def parse_metadata(text):
    def safe_strip(val):
        if val is None:
            return None
        return remove_special_characters(val.strip())

    subject = safe_strip(text.get("subject"))
    sender = safe_strip(text.get("from"))
    to = safe_strip(text.get("to"))
    cc = safe_strip(text.get("cc"))
    bcc = safe_strip(text.get("bcc"))
    date = safe_strip(text.get("date"))
    user = safe_strip(text.get("user"))

    return subject, sender, to, cc, bcc, date, user



def extract_text(text):
    text = text.strip()

    # Remove newlines within sentences (preserve paragraph breaks)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)  # Replace single newlines with space
    text = re.sub(r"\n{2,}", "\n\n", text)  # Ensure paragraphs remain separated by double newlines

    # paragraphs = text.split("\n\n")  # Preserve paragraphs
    sentences = sent_tokenize(text)

    processed_paragraphs = []
    buffer = []
    word_count = 0

    for sent in sentences:
        words = sent.split()
        word_limit = random.randint(60, 400)  # Randomly choose a limit between 60 and 400
        if word_count + len(words) < word_limit:
            buffer.extend(words)
            word_count += len(words)
        else:
            if word_count + len(words) > 400:
                sentence_ends = [i for i, word in enumerate(buffer) if word[-1] in ".?!"]
                split_idx = max(sentence_ends, default=len(buffer) // 2) + 1
                processed_paragraphs.append(" ".join(buffer[:split_idx]).strip())
                buffer = buffer[split_idx:] + words
                word_count = len(buffer)
            else:
                buffer.extend(words)
                processed_paragraphs.append(" ".join(buffer).strip())
                buffer = []
                word_count = 0

    if buffer:
        processed_paragraphs.append(" ".join(buffer).strip())

    processed_paragraphs = [chunk for chunk in processed_paragraphs if chunk]  # Remove blank chunks

    return text, processed_paragraphs


def process_dataset(dataset, sample_size=100000, batch_size=5000, seed=42):
    # Set random seed for reproducibility
    random.seed(seed)

    # sample_size = min(len(dataset), sample_size)
    sample_size = len(dataset)
    sampled_data = dataset.shuffle(seed=seed).select(range(sample_size))

    processed_data = []
    prompt_types = ["gen_full_metadata", "two_step", "enhance"]

    # Ensure equal distribution of prompt types
    num_prompts_per_type = sample_size // len(prompt_types)
    remainder = sample_size % len(prompt_types)
    prompt_distribution = (prompt_types * num_prompts_per_type) + random.sample(prompt_types, remainder)
    random.shuffle(prompt_distribution)  # Shuffle to mix prompt types

    i = 0
    for entry in tqdm(sampled_data, desc="Processing dataset", unit="entry"):
        subject, sender, to, cc, bcc, date, user = parse_metadata(entry)
        text, text_chunks = extract_text(entry["text"])

        if text:
            # {"doc_id": [], "text": [], "metadata": [], "model_name": [], "domain": [], "link": []}
            processed_data.append(
                {
                    "doc_id": generate_content_hash_id(text),
                    "text": text,
                    "metadata": {
                        "subject": subject,
                        "from": sender,
                        "to": to,
                        "cc": cc,
                        "bcc": bcc,
                        "date": date,
                        "user": user
                    },
                    # "text_chunks": text_chunks,
                    "model_name": "human",
                    "domain": "Email & Communications",
                    "link": "https://huggingface.co/datasets/polygraf-ai/enron-mails",
                    # "prompt": prompt,
                    # "prompt_type": prompt_distribution[i],
                    # "prompt_comparison": str(used_chunk_idx),  # Store which chunk index was used
                }
            )

        # When the batch size is met or the last entry is reached
        if (i + 1) % batch_size == 0 or (i + 1) == len(sampled_data):
            # Load the existing dataset from the hub to avoid losing previous data
            try:
                existing_data = datasets.load_dataset(hub_name, split="train")
                combined_data = datasets.concatenate_datasets(
                    [existing_data, datasets.Dataset.from_list(processed_data)]
                )
            except Exception as e:
                print(f"Error loading dataset from hub: {e}")
                combined_data = datasets.Dataset.from_list(processed_data)

            # Push the combined data to the hub
            upload_dataset_to_collection(combined_data, hub_name, collection_title="AID Training Datasets")

            # Clear processed data to release memory
            processed_data.clear()

        i += 1

    # Push the entire processed dataset after all batches
    if processed_data:
        try:
            existing_data = datasets.load_dataset(hub_name, split="train")
            combined_data = datasets.concatenate_datasets([existing_data, datasets.Dataset.from_list(processed_data)])
        except Exception as e:
            print(f"Error loading dataset from hub: {e}")
            combined_data = datasets.Dataset.from_list(processed_data)
        print("Final dataset: ", combined_data)
        upload_dataset_to_collection(combined_data, hub_name, collection_title="AID Training Datasets")


# Load dataset
dataset = datasets.load_dataset("polygraf-ai/enron-mails")["train"]
# column_name = "domain"
# unique_values = dataset.unique(column=column_name)
# print(f"Unique values in column '{column_name}': {unique_values}")
process_dataset(dataset, len(dataset), 50000)  # Adjust batch size if needed


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format: 0ba [00:00, ?ba/s]

README.md:   0%|          | 0.00/435 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
Processing dataset:   2%|▏         | 2483/115374 [00:01<01:14, 1521.42entry/s]c:\Users\minko\OneDrive\Documents\RIT\Fall25\CSCI 788\fact-extraction\cleaning.py:10: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnin

Error decoding text: 'unicodeescape' codec can't decode bytes in position 8-9: malformed \N character escape


...

Creating parquet from Arrow format:   0%|          | 0/116 [00:00<?, ?ba/s]

Processing dataset: 100%|█████████▉| 115216/115374 [01:45<00:00, 1674.62entry/s]

✅ Uploaded dataset: polygraf-ai/aid-enron-mails


Processing dataset: 100%|██████████| 115374/115374 [01:50<00:00, 1040.62entry/s]

ℹ️ Dataset 'polygraf-ai/aid-enron-mails' is already in collection 'AID Training Datasets'
